# Step 1: Install Required Libraries

In [ ]:
!pip install -q langchain langchain-core langchain-community langchain-huggingface huggingface_hub langsmith transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


# Step 2: Import Libraries

In [ ]:
import os
from transformers import pipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_community.llms import HuggingFacePipeline

# Step 3: Set Environment Variables

In [ ]:
os.environ["HUGGINGFACEHUB_API_TOKEN"] = ""
os.environ["LANGCHAIN_API_KEY"] = ""
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "AI Resume Screening System"

# Step 4: Load Hugging Face Model

In [ ]:
hf_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=200
)
llm = HuggingFacePipeline(pipeline=hf_pipeline)

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaF

# Step 5: Input Data

In [ ]:
job_description = '''
We are hiring a Data Scientist.
Required Skills:
Python, Machine Learning, SQL, Pandas, Scikit-learn, Data Visualization
Preferred Tools:
TensorFlow, Tableau, AWS
Experience:
Minimum 2 years in Data Science
'''

strong_resume = '''
John Doe
3 years experience as Data Scientist.
Skills: Python, Machine Learning, SQL, Pandas,
Scikit-learn, TensorFlow, Tableau, AWS, Data Visualization.
'''

average_resume = '''
Jane Smith
2 years experience as Data Analyst.
Skills: Python, SQL, Pandas, Data Visualization, Excel.
Basic Machine Learning knowledge.
'''

weak_resume = '''
Mark Lee
Fresher graduate.
Skills: MS Office, HTML, CSS, Communication.
No Data Science experience.
'''

incorrect_resume = '''
Candidate Name: Unknown
Skills: Singing, Dancing
Experience: None
'''

# Step 6: Create Extraction Chain

In [ ]:
extract_prompt = PromptTemplate.from_template("""
Extract the following from the resume:
1. Skills
2. Experience
3. Tools

Rules:
- Do NOT assume skills not mentioned.
- Return only extracted information.

Resume:
{resume}
""")
extract_chain = extract_prompt | llm

# Step 7: Matching Logic

In [ ]:
job_skills = [
    "python","machine learning","sql",
    "pandas","scikit-learn","data visualization"
]

def match_logic(data):
    extracted_text = data["skills"].lower()
    matched = [skill for skill in job_skills if skill in extracted_text]
    missing = [skill for skill in job_skills if skill not in extracted_text]
    return {"matched": matched, "missing": missing}

match_chain = RunnableLambda(match_logic)

# Step 8: Scoring Logic

In [ ]:
def score_logic(data):
    total = len(job_skills)
    score = int((len(data["matched"]) / total) * 100)
    return {**data, "score": score}

score_chain = RunnableLambda(score_logic)

# Step 9: Explanation Logic

In [ ]:
def explain_logic(data):
    return f"""
Fit Score: {data['score']}/100

Matched Skills:
{data['matched']}

Missing Skills:
{data['missing']}

Explanation:
Candidate suitability is evaluated based on overlap
between resume skills and required job skills.
"""

explain_chain = RunnableLambda(explain_logic)

# Step 10: Full Pipeline Function

In [ ]:
def evaluate_resume(candidate_name, resume):
    print("=" * 60)
    print(f"Evaluating: {candidate_name}")
    print("=" * 60)

    extracted = extract_chain.invoke({"resume": resume})
    print("\n[Extracted Resume Data]")
    print(extracted)

    matched_data = match_chain.invoke({"skills": extracted})
    scored_data = score_chain.invoke(matched_data)
    final_output = explain_chain.invoke(scored_data)

    print("\n[Final Evaluation]")
    print(final_output)
    print("\n")

# Step 11: Run All Test Cases

In [ ]:
evaluate_resume("Strong Candidate", strong_resume)
evaluate_resume("Average Candidate", average_resume)
evaluate_resume("Weak Candidate", weak_resume)
evaluate_resume("Incorrect Debug Candidate", incorrect_resume)

Evaluating: Strong Candidate


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Extracted Resume Data]

Extract the following from the resume:
1. Skills
2. Experience
3. Tools

Rules:
- Do NOT assume skills not mentioned.
- Return only extracted information.

Resume:

John Doe
3 years experience as Data Scientist.
Skills: Python, Machine Learning, SQL, Pandas,
Scikit-learn, TensorFlow, Tableau, AWS, Data Visualization.

 had taken at any hour:, 1/10 (December 9, 1991...Slated, 2001/048): and 1. Skills used within John Donnelll for Data visualization 2. See qualifications for more on both sites above about J-DOT for work listed in application specific position with: Basic software-not all experience in such discipline relevant

[Final Evaluation]

Fit Score: 100/100

Matched Skills:
['python', 'machine learning', 'sql', 'pandas', 'scikit-learn', 'data visualization']

Missing Skills:
[]

Explanation:
Candidate suitability is evaluated based on overlap
between resume skills and required job skills.



Evaluating: Average Candidate


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Extracted Resume Data]

Extract the following from the resume:
1. Skills
2. Experience
3. Tools

Rules:
- Do NOT assume skills not mentioned.
- Return only extracted information.

Resume:

Jane Smith
2 years experience as Data Analyst.
Skills: Python, SQL, Pandas, Data Visualization, Excel.
Basic Machine Learning knowledge.



[Final Evaluation]

Fit Score: 83/100

Matched Skills:
['python', 'machine learning', 'sql', 'pandas', 'data visualization']

Missing Skills:
['scikit-learn']

Explanation:
Candidate suitability is evaluated based on overlap
between resume skills and required job skills.



Evaluating: Weak Candidate


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Extracted Resume Data]

Extract the following from the resume:
1. Skills
2. Experience
3. Tools

Rules:
- Do NOT assume skills not mentioned.
- Return only extracted information.

Resume:

Mark Lee
Fresher graduate.
Skills: MS Office, HTML, CSS, Communication.
No Data Science experience.



[Final Evaluation]

Fit Score: 0/100

Matched Skills:
[]

Missing Skills:
['python', 'machine learning', 'sql', 'pandas', 'scikit-learn', 'data visualization']

Explanation:
Candidate suitability is evaluated based on overlap
between resume skills and required job skills.



Evaluating: Incorrect Debug Candidate


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Extracted Resume Data]

Extract the following from the resume:
1. Skills
2. Experience
3. Tools

Rules:
- Do NOT assume skills not mentioned.
- Return only extracted information.

Resume:

Candidate Name: Unknown
Skills: Singing, Dancing
Experience: None

t

[Final Evaluation]

Fit Score: 0/100

Matched Skills:
[]

Missing Skills:
['python', 'machine learning', 'sql', 'pandas', 'scikit-learn', 'data visualization']

Explanation:
Candidate suitability is evaluated based on overlap
between resume skills and required job skills.



